In [6]:
!pip install "psycopg[binary]" --user --break-system-packages

In [1]:
import sqlite3

# Nasz skrypt DDL dla SQLite
sql_script = """
CREATE TABLE IF NOT EXISTS Czytelnicy (
    ID_Czytelnika INTEGER PRIMARY KEY AUTOINCREMENT,
    Imie TEXT NOT NULL,
    Nazwisko TEXT NOT NULL,
    Ulica TEXT,
    Kod_Pocztowy TEXT,
    Miasto TEXT,
    Data_Zapisu TEXT
);

CREATE TABLE IF NOT EXISTS Autorzy (
    ID_Autora INTEGER PRIMARY KEY AUTOINCREMENT,
    Imie TEXT NOT NULL,
    Nazwisko TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS Kategorie (
    ID_Kategorii INTEGER PRIMARY KEY AUTOINCREMENT,
    Nazwa_Kategorii TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS Ksiazki (
    ID_Ksiazki INTEGER PRIMARY KEY AUTOINCREMENT,
    ID_Autora INTEGER,
    ID_Kategorii INTEGER,
    Tytul TEXT NOT NULL,
    Rok_Wydania INTEGER,
    FOREIGN KEY(ID_Autora) REFERENCES Autorzy(ID_Autora),
    FOREIGN KEY(ID_Kategorii) REFERENCES Kategorie(ID_Kategorii)
);

CREATE TABLE IF NOT EXISTS Wypozyczenia (
    ID_Wypozyczenia INTEGER PRIMARY KEY AUTOINCREMENT,
    ID_Czytelnika INTEGER,
    ID_Ksiazki INTEGER,
    Data_Wypozyczenia TEXT,
    Data_Zwrotu TEXT,
    FOREIGN KEY(ID_Czytelnika) REFERENCES Czytelnicy(ID_Czytelnika),
    FOREIGN KEY(ID_Ksiazki) REFERENCES Ksiazki(ID_Ksiazki)
);
"""

# Tworzymy połączenie (plik biblioteka.db zostanie utworzony automatycznie)
with sqlite3.connect("biblioteka.db") as conn:
    cursor = conn.cursor()
    # Uruchamiamy cały skrypt
    cursor.executescript(sql_script)
    print("✅ Plik biblioteka.db i tabele zostały pomyślnie utworzone!")
    
    # Sprawdzamy, czy tabele fizycznie istnieją
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
    tabele = cursor.fetchall()
    print("Utworzone tabele:", [t[0] for t in tabele])

✅ Plik biblioteka.db i tabele zostały pomyślnie utworzone!
Utworzone tabele: ['Czytelnicy', 'Autorzy', 'Kategorie', 'Ksiazki', 'Wypozyczenia']


In [11]:
import sys
sys.path.append('/home/student15/.local/lib/python3.13/site-packages')

import csv
import psycopg
# 1. Wczytywanie danych z pliku CSV
def wczytaj_dane_csv(sciezka_pliku):
    dane = []
    with open(sciezka_pliku, mode='r', encoding='utf-8') as plik:
        csv_reader = csv.reader(plik)
        next(csv_reader)  # Omijamy nagłówek "Nazwa_Kategorii"
        for wiersz in csv_reader:
            # executemany wymaga krotek, a nasz wiersz ma jeden element, np. ('Fantastyka',)
            dane.append(tuple(wiersz))
    return dane

# 2. Wsadowy import do PostgreSQL
def importuj_do_postgres(kategorie_do_importu, db_config):
    try:
        # Context manager automatycznie zamknie połączenie i zatwierdzi transakcję (commit)
        with psycopg.connect(**db_config) as conn:
            with conn.cursor() as cursor:
                # Sparametryzowane zapytanie (%s) chroni przed SQL Injection
                zapytanie_sql = "INSERT INTO Kategorie (Nazwa_Kategorii) VALUES (%s)"
                
                # Szybki wrzut wszystkich rekordów naraz
                cursor.executemany(zapytanie_sql, kategorie_do_importu)
                
        print(f"✅ Pomyślnie zaimportowano {len(kategorie_do_importu)} rekordów do tabeli Kategorie w PostgreSQL.")
        
    except Exception as e:
        print(f"❌ Wystąpił błąd operacji wsadowej: {e}")

# ==========================================
# KONFIGURACJA I URUCHOMIENIE
# ==========================================

# UZUPEŁNIJ TE DANE SWOIMI POŚWIADCZENIAMI Z PGADMINA:
konfiguracja_bazy = {
    "dbname": "student15db4",      # Brakowało tego klucza! (Nazwę wziąłem z Twojego pgAdmina)
    "user": "student15@laborki.eu",
    "password": "fwtnfhqxdZNP",    # Zostaw swoje hasło
    "host": "127.0.0.1",           # Zmienione z 'host name'
    "port": "5432"                 # Zmienione z 'port number'
}

# Ścieżka do naszego pliku z danymi
plik_csv = 'kategorie.csv'

# Wykonanie logiki
dane = wczytaj_dane_csv(plik_csv)
if dane:
    importuj_do_postgres(dane, konfiguracja_bazy)
else:
    print("Plik CSV jest pusty lub wystąpił problem z jego odczytem.")

❌ Wystąpił błąd operacji wsadowej: connection failed: connection to server at "127.0.0.1", port 5432 failed: KATASTROFALNY:  autoryzacja hasłem nie powiodła się dla użytkownika "student15@laborki.eu"
connection to server at "127.0.0.1", port 5432 failed: KATASTROFALNY:  autoryzacja hasłem nie powiodła się dla użytkownika "student15@laborki.eu"
